# 02 - MACD 和 RSI 因子计算

**功能**: 计算 MACD 和 RSI 指标并相对化

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from scipy import stats

PROJECT_ROOT = Path('../../').resolve()
db_path = PROJECT_ROOT / 'data' / 'invest.db'
conn = sqlite3.connect(db_path)

def 计算 MACD(close, fast=12, slow=26, signal=9):
    exp1 = close.ewm(span=fast, adjust=False).mean()
    exp2 = close.ewm(span=slow, adjust=False).mean()
    dif = exp1 - exp2
    dea = dif.ewm(span=signal, adjust=False).mean()
    macd_hist = 2 * (dif - dea)
    return {'dif': dif, 'dea': dea, 'macd_hist': macd_hist}

def 计算 RSI(close, period=14):
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def 计算分位数 (series, lookback=250):
    return series.rolling(lookback).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100 if len(x) > 10 else np.nan
    )

print("✅ 函数定义完成")

In [ ]:
# 读取数据
daily_df = pd.read_sql_query("SELECT * FROM stock_daily ORDER BY trade_date", conn)
daily_df['trade_date'] = pd.to_datetime(daily_df['trade_date'], format='%Y%m%d')

# 测试股票
test_code = daily_df['ts_code'].iloc[0]
stock_data = daily_df[daily_df['ts_code'] == test_code].sort_values('trade_date').copy()
stock_data.set_index('trade_date', inplace=True)

print(f"测试股票：{test_code}")

In [ ]:
# 计算 MACD
macd = 计算 MACD(stock_data['close'])
stock_data['macd_dif'] = macd['dif']
stock_data['macd_dea'] = macd['dea']
stock_data['macd_hist'] = macd['macd_hist']
stock_data['macd_dif_pct'] = 计算分位数 (macd['dif'])

# 计算 RSI
rsi = 计算 RSI(stock_data['close'])
stock_data['rsi'] = rsi
stock_data['rsi_pct'] = 计算分位数 (rsi)

print("✅ MACD 和 RSI 计算完成")

In [ ]:
# 可视化
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.4, 0.3, 0.3])

# 股价
fig.add_trace(go.Scatter(x=stock_data.index, y=stock_data['close'], name='收盘价'), row=1, col=1)

# MACD
fig.add_trace(go.Scatter(x=stock_data.index, y=macd['dif'], name='DIF', line=dict(color='blue')), row=2, col=1)
fig.add_trace(go.Scatter(x=stock_data.index, y=macd['dea'], name='DEA', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Bar(x=stock_data.index, y=macd['macd_hist'], name='MACD 柱', marker_color=np.where(macd['macd_hist']>0, 'red', 'green')), row=2, col=1)

# RSI
fig.add_trace(go.Scatter(x=stock_data.index, y=rsi, name='RSI', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=3, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=3, col=1)

fig.update_layout(title=f'{test_code} - MACD & RSI', height=800, hovermode='x unified')
fig.show()